# Chapter 5: Gaussian Processes in Remote Sensing

Atmospheric observations are spatially and temporally correlated.
- AOD varies smoothly in space (aerosol plumes have structure)
- Temperature anomalies are persistent over days
- Satellite swaths leave gaps that need interpolation

A Gaussian Process (GP) models these correlations explicitly, enabling:
1. **Spatial interpolation** (kriging) — fill gaps in satellite coverage
2. **Temporal fusion** — combine observations at different times
3. **Covariance estimation** — infer spatial structure of errors
4. **Emulation** — fast surrogate for expensive radiative transfer models


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import cholesky, solve_triangular

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
np.random.seed(42)
print("Ready.")


## 5.1 Spatial Kriging of AOD — Filling Satellite Gaps

MODIS has gaps due to:
- Cloud cover (50–70% of retrievals lost)
- Swath edges
- Sun glint over ocean

We use a 2D GP to interpolate AOD over a 10°×10° domain.

**Physical motivation for the kernel:** Aerosol plumes are spatially correlated
over scales of 100s–1000s km. A Matérn 5/2 kernel with length scale ~2° works well.


In [ ]:
# Simulate a realistic AOD field over a domain
# Then sample it sparsely (like MODIS with cloud gaps) and interpolate with GP

def k_matern52(r, sigma_f=1.0, ell=2.0):
    x = np.sqrt(5)*r/ell
    return sigma_f**2 * (1 + x + x**2/3) * np.exp(-x)

def k_se(r, sigma_f=1.0, ell=1.5):
    return sigma_f**2 * np.exp(-r**2/(2*ell**2))

# True AOD field: sum of a background + two aerosol plumes
x_grid = np.linspace(0, 10, 80)
y_grid = np.linspace(0, 10, 80)
X_g, Y_g = np.meshgrid(x_grid, y_grid)

# True AOD: background + two plumes
AOD_true = (0.15
            + 0.6 * np.exp(-((X_g-3)**2 + (Y_g-4)**2)/4)   # urban plume
            + 0.3 * np.exp(-((X_g-7)**2 + (Y_g-7)**2)/2))  # dust plume
AOD_true = np.clip(AOD_true, 0.05, None)

# Sample: random locations (MODIS-like, 40% coverage with cloud gaps)
np.random.seed(5)
n_obs = 200
x_obs = np.random.uniform(0, 10, n_obs*2)
y_obs = np.random.uniform(0, 10, n_obs*2)

# Simulate cloud mask (remove observations in cloudy areas)
cloud_mask = ~(((x_obs-2)**2 + (y_obs-8)**2 < 4) |   # cloud 1
               ((x_obs-6)**2 + (y_obs-3)**2 < 3))     # cloud 2
x_obs = x_obs[cloud_mask][:n_obs]
y_obs = y_obs[cloud_mask][:n_obs]

# True AOD at observation locations (bilinear interpolation)
from scipy.interpolate import RegularGridInterpolator
interp = RegularGridInterpolator((y_grid, x_grid), AOD_true, method='linear')
AOD_obs_true = interp(np.c_[y_obs, x_obs])

# Add noise
sigma_n = 0.03
AOD_obs = AOD_obs_true + sigma_n * np.random.randn(len(x_obs))
AOD_obs = np.clip(AOD_obs, 0.01, None)

print(f"Observations: {n_obs} (out of 6400 grid points = {100*n_obs/6400:.0f}%)")

# ── GP regression in 2D ─────────────────────────────────────────
# Compute distance matrix between observations
dist_obs = np.sqrt((x_obs[:,None]-x_obs[None,:])**2 + (y_obs[:,None]-y_obs[None,:])**2)

sigma_f_gp, ell_gp = 0.25, 1.8
K_obs = k_matern52(dist_obs, sigma_f_gp, ell_gp) + (sigma_n**2 + 1e-6)*np.eye(n_obs)

# Predict on the full grid
x_flat = X_g.ravel(); y_flat = Y_g.ravel()
dist_pred_obs = np.sqrt((x_flat[:,None]-x_obs[None,:])**2 + (y_flat[:,None]-y_obs[None,:])**2)
K_pred_obs = k_matern52(dist_pred_obs, sigma_f_gp, ell_gp)

L = cholesky(K_obs, lower=True)
alpha = solve_triangular(L.T, solve_triangular(L, AOD_obs, lower=True))

AOD_mean = 0.2   # prior mean
mu_pred = AOD_mean + K_pred_obs @ alpha
mu_pred = mu_pred.reshape(X_g.shape)

V_pred = k_matern52(0, sigma_f_gp, ell_gp) - np.sum(solve_triangular(L, K_pred_obs.T, lower=True)**2, axis=0)
std_pred = np.sqrt(np.clip(V_pred, 0, None)).reshape(X_g.shape)

print(f"GP prediction complete. Mean RMSE: {np.sqrt(np.mean((mu_pred - AOD_true)**2)):.4f}")


In [ ]:
# Visualise: true / observations / GP prediction / uncertainty
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

vmin, vmax = 0.05, 0.9

im0 = axes[0,0].pcolormesh(X_g, Y_g, AOD_true, cmap="YlOrRd", vmin=vmin, vmax=vmax)
axes[0,0].set_title("True AOD field"); plt.colorbar(im0, ax=axes[0,0], label="AOD")

axes[0,1].scatter(x_obs, y_obs, c=AOD_obs, cmap="YlOrRd", s=8, vmin=vmin, vmax=vmax)
axes[0,1].add_patch(plt.Circle((2,8), 2, fill=True, color="lightblue", alpha=0.5, label="Cloud 1"))
axes[0,1].add_patch(plt.Circle((6,3), np.sqrt(3), fill=True, color="lightblue", alpha=0.5, label="Cloud 2"))
axes[0,1].set_title(f"MODIS-like Observations
({n_obs} pixels, cloud gaps visible)")
axes[0,1].legend(fontsize=8, frameon=False)

im2 = axes[1,0].pcolormesh(X_g, Y_g, mu_pred, cmap="YlOrRd", vmin=vmin, vmax=vmax)
axes[1,0].contour(X_g, Y_g, mu_pred, levels=8, colors="white", linewidths=0.5, alpha=0.4)
axes[1,0].set_title("GP Kriging Prediction
(gap-filled AOD)"); plt.colorbar(im2, ax=axes[1,0], label="AOD")

im3 = axes[1,1].pcolormesh(X_g, Y_g, std_pred, cmap="Blues", vmin=0, vmax=0.15)
axes[1,1].set_title("GP Prediction Uncertainty (1σ)
(largest in cloud gaps)"); plt.colorbar(im3, ax=axes[1,1], label="σ AOD")

for ax in axes.ravel():
    ax.set_xlabel("Longitude (°)"); ax.set_ylabel("Latitude (°)")

plt.suptitle("GP Kriging: Gap-Filling AOD Observations Under Cloud Cover", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 5.2 Temporal GP — Tracking Aerosol Events

Aerosol events (dust storms, biomass burning) evolve over days to weeks.
A GP in time models this evolution and propagates uncertainty when observations are missing.


In [ ]:
# Temporal GP: AOD time series at a single location
# Simulate a dust event passing over a station
np.random.seed(7)
t_days = np.linspace(0, 30, 300)   # 30 days, 0.1-day resolution

# True AOD: background + Gaussian dust event
AOD_bg   = 0.12
AOD_dust = 0.8 * np.exp(-((t_days - 12)**2)/8)
AOD_sea  = 0.15 * np.sin(2*np.pi*t_days/7)   # weekly cycle (sea breeze)
AOD_true_t = AOD_bg + AOD_dust + AOD_sea + 0.02*np.random.randn(300)
AOD_true_t = np.clip(AOD_true_t, 0.02, None)

# Sparse observations (e.g., AERONET, daily at best)
t_obs_mask = np.random.choice(len(t_days), 35, replace=False)
t_obs_mask = np.sort(t_obs_mask)
t_obs = t_days[t_obs_mask]
AOD_t_obs = AOD_true_t[t_obs_mask] + 0.03*np.random.randn(len(t_obs_mask))

# Gap periods
t_obs_masked = t_obs.copy()
# Remove observations during days 15-20 (bad weather at station)
keep = ~((t_obs > 14) & (t_obs < 20))
t_obs = t_obs[keep]; AOD_t_obs = AOD_t_obs[keep]

# Quasi-periodic kernel for temporal GP (smooth + weekly)
def k_qp_time(t1, t2, sf=0.3, ell_smooth=5.0, period=7.0, ell_per=1.5):
    dt = np.abs(t1 - t2)
    return sf**2 * np.exp(-dt**2/(2*ell_smooth**2)) * np.exp(-2*np.sin(np.pi*dt/period)**2/ell_per**2)

def build_K(t1, t2):
    return np.array([[k_qp_time(a, b) for b in t2] for a in t1])

K_tt = build_K(t_obs, t_obs) + 0.03**2 * np.eye(len(t_obs))
K_pt = build_K(t_days, t_obs)

L_t = cholesky(K_tt, lower=True)
alpha_t = solve_triangular(L_t.T, solve_triangular(L_t, AOD_t_obs, lower=True))

mu_t = 0.15 + K_pt @ alpha_t
V_t  = k_qp_time(0,0) - np.sum(solve_triangular(L_t, K_pt.T, lower=True)**2, axis=0)
std_t = np.sqrt(np.clip(V_t, 0, None))

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(t_days, mu_t-2*std_t, mu_t+2*std_t, color="#3B8BD4", alpha=0.15, label="±2σ")
ax.fill_between(t_days, mu_t-std_t,   mu_t+std_t,   color="#3B8BD4", alpha=0.30, label="±1σ")
ax.plot(t_days, mu_t, "#3B8BD4", lw=2.5, label="GP mean")
ax.plot(t_days, AOD_true_t, "#1D9E75", lw=2, ls="--", alpha=0.6, label="True AOD")
ax.scatter(t_obs, AOD_t_obs, color="black", s=50, zorder=5, label="Observations")
ax.axvspan(14, 20, color="#D85A30", alpha=0.1, label="Data gap (bad weather)")
ax.set_xlabel("Day of month")
ax.set_ylabel("AOD at 550 nm")
ax.set_title("Temporal GP: AOD Event Reconstruction from Sparse Observations
"
             "(Quasi-periodic kernel captures dust event + weekly cycle)")
ax.legend(frameon=False, fontsize=9)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

peak_idx = np.argmax(mu_t)
print(f"GP-estimated dust event peak: day {t_days[peak_idx]:.1f}, AOD = {mu_t[peak_idx]:.3f}")
print(f"True peak: day {t_days[np.argmax(AOD_true_t)]:.1f}, AOD = {AOD_true_t.max():.3f}")


In [ ]:
print("Chapter 5 complete!")
print()
print("GP applications in atmospheric remote sensing:")
print("  Spatial:   Kriging to fill cloud gaps in AOD / precipitation / SST")
print("  Temporal:  Tracking aerosol events, interpolating between overpasses")
print("  Emulation: Replace expensive RT code with fast GP surrogate for MCMC")
print("  Fusion:    Combine satellite + ground obs + model at different resolutions")
print()
print("Key kernel choices for atmospheric data:")
print("  Matérn 5/2: smooth spatial fields (temperature, humidity)")
print("  Matérn 3/2: rougher fields (precipitation, aerosols)")
print("  Quasi-periodic: recurring signals (diurnal cycle, seasonal cycle)")
print("  Rational quadratic: multi-scale variability (turbulence + synoptic)")
